<a href="https://colab.research.google.com/github/seanyi-creator/ba-305-project/blob/main/02_logistic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workstream 1b — Logistic regression
**Owner:** Sean Yi · **Question:** can the details on a permit application predict whether the project will fail to finish?

Logistic regression is method 1 of 2. It was chosen because permit staff need to know *why* a permit is flagged: each feature gets one coefficient with a clear direction, and exp(coefficient) says how much it multiplies the odds of not finishing. Its limitation is that it can only draw a straight-line boundary, which is what the decision tree (workstream 2) is there to test.

Steps: load the shared data → tune two hyperparameters with 5-fold cross-validation → diagnose the fit → score once on the test set against the baselines → interpret the coefficients.

In [ ]:
# --- Sean: setup ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score,
                             RocCurveDisplay, ConfusionMatrixDisplay)
pd.set_option('display.max_columns', 50)

## Shared data (copied unchanged from `01_permits_prep.ipynb`)

In [ ]:
# --- Sean: SHARED CELL — load, target, filing-year filter, leakage drops, features, split, preprocessor ---
# Copy this whole cell into your notebook, then run:
#   X_train, X_test, y_train, y_test, groups_train, preprocessor = get_permit_data()
# and cross-validate with cv=make_cv() and .fit(X_train, y_train, groups=groups_train)
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PERMITS_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2022/2022-07-05/sf_permits.csv"

# Statuses where the outcome is final. 'complete' = 0; every other final status = 1 (did not finish).
FINAL_STATUS = {'complete': 0, 'expired': 1, 'cancelled': 1, 'withdrawn': 1,
                'disapproved': 1, 'revoked': 1, 'denied': 1}
YEAR_MIN, YEAR_MAX = 2000, 2013   # filing years kept (see Step 2)
RANDOM_STATE = 42                 # everyone uses this, so the split is identical

# Columns recorded AFTER filing (leak the outcome) or that are only labels for a case (see Step 3).
LEAKAGE_COLS = ['status', 'status_date', 'completed_date', 'issued_date', 'permit_expiration_date',
                'first_construction_document_date', 'revised_cost']
ID_COLS = ['permit_number', 'record_id', 'block', 'lot', 'street_number', 'street_number_suffix',
           'street_name', 'street_suffix', 'unit', 'unit_suffix', 'location', 'description',
           'permit_creation_date', 'filed_date', 'date', 'zipcode', 'tidf_compliance',
           'permit_type', 'existing_construction_type', 'proposed_construction_type']
# description: free text, left out of the baseline feature set (an optional extension).
# zipcode: overlaps with neighborhood; permit_type / construction_type codes: duplicates of their text labels.
# tidf_compliance: 100% missing.

NUMERIC = ['log_estimated_cost', 'cost_is_placeholder', 'plansets',
           'existing_units', 'proposed_units', 'unit_change',
           'number_of_existing_stories', 'number_of_proposed_stories', 'story_change',
           'filing_year', 'filing_month']
CATEGORICAL = ['permit_type_definition', 'existing_use', 'proposed_use', 'use_changes',
               'existing_construction_type_description', 'proposed_construction_type_description',
               'structural_notification', 'site_permit', 'fire_only_permit',
               'voluntary_soft_story_retrofit', 'neighborhoods_analysis_boundaries',
               'supervisor_district']


def load_permits(url=PERMITS_URL):
    """Raw file, as downloaded."""
    return pd.read_csv(url, low_memory=False)


def resolved_permits(raw):
    """Keep permits with a final outcome, filed in YEAR_MIN..YEAR_MAX, and add the target y."""
    df = raw[raw['status'].isin(FINAL_STATUS)].copy()
    df['filed'] = pd.to_datetime(df['filed_date'])
    df = df[df['filed'].dt.year.between(YEAR_MIN, YEAR_MAX)]
    df['y'] = df['status'].map(FINAL_STATUS)          # 1 = did not finish
    return df


def add_features(df):
    """Features that are all known on the day the permit is filed."""
    out = df.copy()
    cost = out['estimated_cost']
    out['cost_is_placeholder'] = (cost <= 1).astype(int)          # $0/$1 entered as a placeholder
    out['log_estimated_cost'] = np.log10(cost.where(cost > 1))     # skewed: $1 to $500M+, so use log10
    out['unit_change'] = out['proposed_units'] - out['existing_units']
    out['story_change'] = out['number_of_proposed_stories'] - out['number_of_existing_stories']
    both_known = out['existing_use'].notna() & out['proposed_use'].notna()
    out['use_changes'] = (out['existing_use'] != out['proposed_use']).map({True: 'yes', False: 'no'}).where(both_known)
    out['filing_year'] = out['filed'].dt.year
    out['filing_month'] = out['filed'].dt.month
    out['supervisor_district'] = out['supervisor_district'].astype('Int64').astype(str).replace('<NA>', np.nan)
    for c in ['structural_notification', 'site_permit', 'fire_only_permit', 'voluntary_soft_story_retrofit']:
        out[c] = out[c].fillna('N')                                # blank means "not flagged"
    return out


def make_preprocessor():
    """Fit inside a Pipeline, so imputation and scaling learn from training folds only."""
    numeric = Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True)),  # median + a 'was missing' flag
        ('scale', StandardScaler()),
    ])
    categorical = Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50)),  # rare levels pooled
    ])
    return ColumnTransformer([('num', numeric, NUMERIC), ('cat', categorical, CATEGORICAL)])


def batch_id(df):
    """Permits filed for the same lot on the same day are one project's batch (see Step 5)."""
    return df['block'].astype(str) + '-' + df['lot'].astype(str) + '-' + df['filed_date'].astype(str)


def make_cv():
    """5-fold CV that keeps each batch inside one fold and keeps the class mix similar."""
    return StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def get_permit_data(raw=None, test_size=0.2):
    """One call for teammates: same rows, features, grouped split and preprocessor for everyone."""
    raw = load_permits() if raw is None else raw
    df = add_features(resolved_permits(raw)).reset_index(drop=True)
    X, y, groups = df[NUMERIC + CATEGORICAL], df['y'], batch_id(df)
    train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=test_size,
                                                 random_state=RANDOM_STATE).split(X, y, groups))
    return (X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx],
            groups.iloc[train_idx], make_preprocessor())

In [ ]:
# --- Sean: load the shared split ---
X_train, X_test, y_train, y_test, groups_train, preprocessor = get_permit_data()
print(f'Train: {len(X_train):,} permits · Test: {len(X_test):,} permits · did not finish: {y_train.mean():.1%}')

## Step 1 — Tune two hyperparameters
- **`C`** controls how strongly the coefficients are shrunk toward 0 (regularization). Small `C` = a simpler model that may underfit; large `C` = fewer restrictions that may overfit. Swept over 7 values on a log scale, 0.001 to 100.
- **`class_weight`**: `None` treats both classes equally; `'balanced'` gives the minority class ("did not finish", 36%) more weight, so the model pays more attention to failed permits.

Scored by ROC AUC (how well the model ranks risky permits above safe ones, across all cutoffs) with 5-fold cross-validation **on the training set only**, grouped so a project's batch of permits never sits in both the training and validation folds. Because the preprocessor sits inside the pipeline, imputation and scaling are re-fit inside each fold.

In [ ]:
# --- Sean: Step 1 — grid search over C and class_weight ---
logit_pipe = Pipeline([('prep', preprocessor),
                       ('model', LogisticRegression(max_iter=3000))])
param_grid = {'model__C': [0.001, 0.01, 0.1, 0.3, 1, 10, 100],
              'model__class_weight': [None, 'balanced']}
cv = make_cv()   # keeps each lot-and-day batch inside one fold
logit_search = GridSearchCV(logit_pipe, param_grid, cv=cv, scoring='roc_auc',
                            return_train_score=True, n_jobs=-1)
logit_search.fit(X_train, y_train, groups=groups_train)

results = pd.DataFrame(logit_search.cv_results_)
grid_table = results.pivot_table(index='param_model__C', columns='param_model__class_weight',
                                 values='mean_test_score', dropna=False).round(4)
grid_table.columns = ['class_weight=balanced' if c == 'balanced' else 'class_weight=None' for c in grid_table.columns]
print('Best:', logit_search.best_params_, '· CV AUC', round(logit_search.best_score_, 4))
grid_table

**Reading the grid:** if the best `C` sits at the edge of the range (0.001 or 100), widen the range and re-run (the directives ask for this). `class_weight` barely changes AUC, which is expected: AUC measures ranking, and reweighting mostly shifts the probabilities up or down. Where reweighting matters is recall at a fixed cutoff (Step 3).

## Step 2 — Is it overfitting or underfitting?
Training AUC vs. cross-validated AUC across `C`. A widening gap at large `C` would mean overfitting; both curves low and flat means the model is too simple for the pattern (underfitting).

In [ ]:
# --- Sean: Step 2 — fit diagnosis curve ---
best_cw = logit_search.best_params_['model__class_weight']
sub = results[results['params'].apply(lambda p: p['model__class_weight'] == best_cw)].sort_values('param_model__C')
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(sub['param_model__C'].astype(float), sub['mean_train_score'], marker='o', label='training AUC')
ax.plot(sub['param_model__C'].astype(float), sub['mean_test_score'], marker='o', label='cross-validated AUC')
ax.set_xscale('log'); ax.set_xlabel('C (log scale; larger = less regularization)'); ax.set_ylabel('ROC AUC')
ax.set_title(f'Training and CV AUC stay close: no overfitting (class_weight={best_cw})')
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()
sub[['param_model__C', 'mean_train_score', 'mean_test_score']].round(4)

**Reading the curve:** the two lines track each other closely, so the model is not memorizing the training permits. The ceiling around the CV score suggests a *linear* model has extracted most of what it can, which is the case for trying a tree, whose boundaries can bend around interactions (for example "large cost **and** new construction").

## Step 3 — Test-set results vs. the baselines
The tuned pipeline is scored **once** on the held-out test permits. At the default 0.5 cutoff (the threshold is chosen properly, with costs, in workstream 2).

In [ ]:
# --- Sean: Step 3 — score the tuned model on the test set ---
best_logit = logit_search.best_estimator_
prob = best_logit.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

rows = []
for name, strategy in [('Always "completed"', 'most_frequent'), ('Random guess (stratified)', 'stratified')]:
    d = DummyClassifier(strategy=strategy, random_state=RANDOM_STATE).fit(X_train, y_train)
    dp = d.predict(X_test)
    rows.append({'model': name, 'ROC AUC': roc_auc_score(y_test, d.predict_proba(X_test)[:, 1]),
                 'accuracy': accuracy_score(y_test, dp),
                 'recall (did not finish)': recall_score(y_test, dp, zero_division=0),
                 'precision (did not finish)': precision_score(y_test, dp, zero_division=0)})
rows.append({'model': 'Logistic regression (cutoff 0.5)', 'ROC AUC': roc_auc_score(y_test, prob),
             'accuracy': accuracy_score(y_test, pred),
             'recall (did not finish)': recall_score(y_test, pred),
             'precision (did not finish)': precision_score(y_test, pred)})
test_table = pd.DataFrame(rows).round(3)
test_table

In [ ]:
# --- Sean: Step 3 — ROC curve and confusion matrix ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
RocCurveDisplay.from_predictions(y_test, prob, name='Logistic regression', ax=axes[0])
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='random guessing (AUC 0.5)'); axes[0].legend(loc='lower right')
axes[0].set_title('Ranks risky permits well above chance, far from perfect')
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['completed (0)', 'did not finish (1)'],
                                        cmap='Blues', ax=axes[1], colorbar=False)
axes[1].set_title('Test set at cutoff 0.5')
plt.tight_layout(); plt.show()

## Step 4 — What drives the model?
Each coefficient is the change in the log-odds of **not finishing** for a one-unit increase in that feature, holding the others fixed. exp(coefficient) turns it into a multiplier on the odds: above 1 = higher risk, below 1 = lower risk.

Two notes for reading the table:
- Numeric features were **standardized**, so their coefficients are per one standard deviation and can be compared with each other by size.
- Each category level (for example `permit_type_definition = demolitions`) gets its own 0/1 column. Its coefficient compares permits **with** that level to permits without it, holding everything else fixed.

In [ ]:
# --- Sean: Step 4 — coefficient table ---
names = best_logit.named_steps['prep'].get_feature_names_out()
coef = best_logit.named_steps['model'].coef_[0]
coefs = pd.DataFrame({'feature': [n.replace('num__', '').replace('cat__', '') for n in names],
                      'coefficient': coef, 'exp(coefficient)': np.exp(coef)})
coefs['abs'] = coefs['coefficient'].abs()
top = coefs.sort_values('abs', ascending=False).head(15).drop(columns='abs').round(3)
top

In [ ]:
# --- Sean: Step 4 — chart of the 15 strongest effects ---
plot_df = top.sort_values('coefficient')
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(plot_df['feature'], plot_df['coefficient'],
        color=np.where(plot_df['coefficient'] > 0, '#c0581a', '#3b6ea5'))
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('coefficient (log-odds of NOT finishing)')
ax.set_title('Orange raises the risk of not finishing; blue lowers it')
plt.tight_layout(); plt.show()

In [ ]:
# --- Sean: Step 4 — estimated cost in plain words (per 10x increase in cost) ---
# log_estimated_cost is log10(cost), then standardized. One standard deviation of log10(cost) is
# 'sd' units, so a 10x increase in cost (log10 + 1) is 1/sd standard deviations.
sd = best_logit.named_steps['prep'].named_transformers_['num'].named_steps['scale'].scale_[NUMERIC.index('log_estimated_cost')]
b = coefs.loc[coefs['feature'] == 'log_estimated_cost', 'coefficient'].iloc[0]
print(f'A 10x higher estimated cost multiplies the odds of not finishing by {np.exp(b / sd):.2f}, holding the rest fixed.')

**Caution on the table above:** several of the largest coefficients belong to uncommon building uses (each has at least 50 permits, but often not many more), so their estimates are noisy. Lean on effects that are both large and common. The cell below lists the numeric features, which apply to every permit.

In [ ]:
# --- Sean: Step 4 — numeric features only (per one standard deviation) ---
coefs[coefs['feature'].isin(NUMERIC)].drop(columns='abs').sort_values('coefficient').round(3)

**For the report and the talk:** pick the three to five effects in the chart that a permit officer would care about and write each as one plain sentence, for example *"Holding the rest fixed, a demolition permit has about X times the odds of not finishing of an alteration permit."* If a top feature makes no sense, check it for leakage before you trust it (§5.5).

**Hand-off to workstream 2:** the best settings printed in Step 1 go into the `LOGISTIC_BEST` line of `03_tree_decision.ipynb`, so the two models are compared on the same test permits.

In [ ]:
# --- Sean: values to hand to workstream 2 ---
print('LOGISTIC_BEST =', {k.replace('model__', ''): v for k, v in logit_search.best_params_.items()})